### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/71/zf2cv4817r7fbx4dw929lzq80000gn/T/ipykernel_18698/3905054165.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """
    Process all PDF files in the specified directory
    and return a list of Document objects.

    Args:
        pdf_directory (str): Path to the directory containing PDF files.
    """
    documents = []
    pdf_directory = Path(pdf_directory)

    # Find all PDF files
    pdf_files = list(pdf_directory.glob("*.pdf"))

    print(f"📁 Found {len(pdf_files)} PDF files in: {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"↗️ Processing PDF file: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            pdf_docs = loader.load()

            # Add source information
            for doc in pdf_docs:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = ".pdf"

            # Add PDF documents to the main list
            documents.extend(pdf_docs)

            print(f"✅ Successfully processed {len(pdf_docs)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"❌ Error processing {pdf_file}: {e}")

    return documents

# Call the function
all_pdf_documents = process_all_pdfs("../Data/pdf_files")

print(f"\n📚 Total PDF pages loaded: {len(all_pdf_documents)}")

📁 Found 5 PDF files in: ../Data/pdf_files
↗️ Processing PDF file: ../Data/pdf_files/2-Transformers-Basic-to-Advanced.pdf
✅ Successfully processed 25 pages from 2-Transformers-Basic-to-Advanced.pdf
↗️ Processing PDF file: ../Data/pdf_files/1-Embeddings-Basic-to-Advanced.pdf
✅ Successfully processed 22 pages from 1-Embeddings-Basic-to-Advanced.pdf
↗️ Processing PDF file: ../Data/pdf_files/resume.pdf
✅ Successfully processed 1 pages from resume.pdf
↗️ Processing PDF file: ../Data/pdf_files/3-RAG-Basic-to-Advanced.pdf
✅ Successfully processed 21 pages from 3-RAG-Basic-to-Advanced.pdf
↗️ Processing PDF file: ../Data/pdf_files/4-Object-Detection-Basic-to-Advanced.pdf
✅ Successfully processed 23 pages from 4-Object-Detection-Basic-to-Advanced.pdf

📚 Total PDF pages loaded: 92


In [3]:
## Text Splitting

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks.

    Args:
        documents (list): List of Document objects to be split.
        chunk_size (int): Size of each chunk in characters.
        chunk_overlap (int): Overlap size between chunks in characters.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"📄 Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print("Example Chunk Metadata:", split_docs[0].metadata)
        print(f"✅ First chunk preview: {split_docs[0].page_content[:200]}...")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

📄 Split 92 documents into 221 chunks.
Example Chunk Metadata: {'producer': 'Skia/PDF m141', 'creator': 'Chromium', 'creationdate': '2026-10-22T10:53:25+00:00', 'title': 'Transformers', 'moddate': '2026-10-22T10:53:25+00:00', 'subject': 'The architecture behind GPT, BERT, Claude and modern AI, built up from tokens and attention to a working GPT, Hugging Face models and the techniques used in production.', 'author': 'Study guide', 'source': '2-Transformers-Basic-to-Advanced.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'file_type': '.pdf'}
✅ First chunk preview: Trnsformers
The rchitecture behind GPT, BERT, Clude nd
modern AI, built up from tokens nd ttention to 
working GPT, Hugging Fce models nd the techniques
used in production.
From the bsics to...


In [5]:
chunks

[Document(metadata={'producer': 'Skia/PDF m141', 'creator': 'Chromium', 'creationdate': '2026-10-22T10:53:25+00:00', 'title': 'Transformers', 'moddate': '2026-10-22T10:53:25+00:00', 'subject': 'The architecture behind GPT, BERT, Claude and modern AI, built up from tokens and attention to a working GPT, Hugging Face models and the techniques used in production.', 'author': 'Study guide', 'source': '2-Transformers-Basic-to-Advanced.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'file_type': '.pdf'}, page_content='Tr\ue02cnsformers\nThe \ue02crchitecture behind GPT, BERT, Cl\ue02cude \ue02cnd\nmodern AI, built up from tokens \ue02cnd \ue02cttention to \ue02c\nworking GPT, Hugging F\ue02cce models \ue02cnd the techniques\nused in production.\nFrom the b\ue02csics to \ue02cdv\ue02cnced techniques, with expl\ue02cn\ue02ctions \ue02cnd working code.\n\ue072\ue094 Found\ue02ctions: why Tr\ue02cnsformers, tokeniz\ue02ction, embeddings \ue02cnd position\ue02cl encoding\n\ue073\ue094 Atte

### Embedding and vectorStoreDB

In [6]:
%pip install sentence-transformers

/Users/rakshithdheerreddykunduru/Github/Python-Projects/Learn RAG/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/rakshithdheerreddykunduru/Github/Python-Projects/Learn RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 12:19:59.822000 18698 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [8]:
class EmbeddingManager:
    """ Handles Document Embeddings and Similarity Search using Sentence Transformers and ChromaDB. """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Initialize the EmbeddingManager with a specified Sentence Transformer model. 
        Args:
            model_name (str): The name of the Sentence Transformer model to use.
            Hugging face Model for sentence embeddings can be found here: https://huggingface.co/sentence-transformers
        """
        self.model_name = model_name
        self.model = None
        self.load_model() # Going to load the model when the class is initialized

    def load_model(self):
        """ Load the Sentence Transformer model. """
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ Successfully loaded model: {self.model.get_sentence_embedding_dimension()} dimensions for {self.model_name}")
        except Exception as e:
            print(f"❌ Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to generate embeddings for.
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension).
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        print(f"🔹 Generating embeddings for {len(texts)} texts using model: {self.model_name}")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"✅ Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_sentence_embedding_dimension(self) -> int:
        """ Get the embedding dimension of the loaded model.
        Returns:
            int: The dimension of the embeddings produced by the model.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        return self.model.get_sentence_embedding_dimension()
    
## Intialize the EmbeddingManager

embedding_manager = EmbeddingManager()
embedding_manager

    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17588.69it/s]


✅ Successfully loaded model: 384 dimensions for all-MiniLM-L6-v2


/var/folders/71/zf2cv4817r7fbx4dw929lzq80000gn/T/ipykernel_18698/713290512.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Successfully loaded model: {self.model.get_sentence_embedding_dimension()} dimensions for {self.model_name}")


### Vector Store

In [22]:
class VectorStore:
    """ Manages a vector store for document embeddings using ChromaDB. """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../Data/chroma_db"):
        """ Initialize the VectorStore with a specified collection name and persistence directory.
        Args:
            collection_name (str): The name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the ChromaDB database.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store() # Intlizing the vecotor store when the class is initialized

    def initialize_store(self):
        """ Initialize the ChromaDB client and collection. """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing PDF document embeddings for RAG."}
                )
            print(f"✅ Successfully initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
            print(f"🔹 Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"❌ Error initializing ChromaDB: {e}")
            raise 
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """ Add documents and their embeddings to the ChromaDB collection.
        Args:
            documents (List[Document]): List of Document objects to add.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        # Prepare data for ChromaDB
        ids = []
        metadata = []
        documents_texts = [] 
        embeddings_list = []       

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            # Prepare metadata and text for ChromaDB
            doc_metadata = dict(doc.metadata)
            doc_metadata["doc_index"] = i
            doc_metadata["content_length"] = len(doc.page_content)
            
            metadata.append(doc.metadata)

            # Docuements content
            documents_texts.append(doc.page_content)
            # Embeddings
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB

            try:
                # Add to ChromaDB collection
                self.collection.add(
                    ids=ids,
                    metadatas=metadata,
                    documents=documents_texts,
                    embeddings=embeddings_list
                )
                print(f"Sucessfully added {len(documents)} documents to ChromaDB collection: {self.collection_name}")
                print(f"✅ Added document {i+1}/{len(documents)} to ChromaDB collection with ID: {doc_id}")
            except Exception as e:
                print(f"❌ Error adding document {i+1} to ChromaDB: {e}")
                raise

vector_store = VectorStore()
vector_store
        

✅ Successfully initialized ChromaDB collection: pdf_documents at ../Data/chroma_db
🔹 Existing documents in the collection: 0


In [23]:
### COnvert the chunks into embeddings and add them to the vector store
texts = [doc.page_content for doc in chunks]

## Geneerate the emebeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Add the documents and their embeddings to the vector store
vector_store.add_documents(chunks, embeddings)

🔹 Generating embeddings for 221 texts using model: all-MiniLM-L6-v2


Batches: 100%|██████████| 7/7 [00:01<00:00,  5.87it/s]


✅ Generated embeddings with shape: (221, 384)
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 1/221 to ChromaDB collection with ID: c2f25cab-88af-41d8-9f35-64d458fbdd70
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 2/221 to ChromaDB collection with ID: ecffdf74-b944-49f6-9fcf-341664de6b8e
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 3/221 to ChromaDB collection with ID: 5a0b29e9-c2e0-43ee-9cb5-aca8bc8d3c61
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 4/221 to ChromaDB collection with ID: c33f0d4e-0b01-42b0-bb17-7c9dbac4014f
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 5/221 to ChromaDB collection with ID: 7d135932-337c-4a38-b6de-9d76df73412e
Sucessfully added 221 documents to ChromaDB collection: pdf_documents
✅ Added document 6/221 to ChromaDB collection with ID: 894dda1e-c204-4080-

### Retriever Pipeline from Vector Store

In [46]:
class RAGRetriever:
    """ Implements a simple RAG retriever using ChromaDB and Sentence Transformers. """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """ Initialize the RAGRetriever with a vector store and embedding manager.
        Args:
            vector_store (VectorStore): The vector store for document embeddings.
            embedding_manager (EmbeddingManager): The embedding manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """ Retrieve the top_k most relevant documents for a given query.
        Args:
            query (str): The input query string.
            top_k (int): Number of top documents to retrieve.
            score_threshold (float): Minimum similarity score for a document to be included in the results.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
        """
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # Retrieve documents from the vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            # Process results to include score and metadata
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]  # Assuming distances are returned
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score <= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                    print(f"✅ Retrieved document {i+1}: ID={doc_id}, Score={similarity_score:.4f}, Distance={distance:.4f}")
                    print(f"Content Preview: {document[:200]}...")  # Print first 200 characters of the document
                    print(f"Metadata: {metadata}\n")
            else:
                 print(f"⚠️ Document {i+1} below score threshold: ID={doc_id}, Score={similarity_score:.4f}, Distance={distance:.4f}")
            return retrieved_docs
                
        except Exception as e:
            print(f"❌ Error retrieving documents: {e}")
            return []

        
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

In [47]:
rag_retriever

In [48]:
rag_retriever.retrieve("What is the attention all you need?", top_k=3, score_threshold=0.5)

🔹 Generating embeddings for 1 texts using model: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.58it/s]

✅ Generated embeddings with shape: (1, 384)
✅ Retrieved document 1: ID=b6e97c8d-4a56-409a-8f64-45c179346450, Score=-0.4233, Distance=1.4233
Content Preview: Psiog Digital Feb 2022 – Sep 2022
Full Stack Engineer Trainee Chennai, India
• Completed a rigorous 6-month full stack development training programme.
• Built a Movie Recommendation App using C#, ASP....
Metadata: {'file_type': '.pdf', 'creator': 'Writer', 'page': 0, 'total_pages': 1, 'source': 'resume.pdf', 'author': 'Un-named', 'creationdate': '2026-10-21T17:39:43+01:00', 'page_label': '1', 'producer': 'LibreOffice 24.2'}

✅ Retrieved document 2: ID=9fda0f7b-eafc-408a-b9a0-4a7bd644ef2c, Score=-0.4643, Distance=1.4643
Content Preview: • Designed prompts and response handling to keep answers relevant and accurate across multi-turn conversations.
Review Sentiment Analysis | Python, NLP
• Developed an NLP pipeline that cleans, tokenis...
Metadata: {'total_pages': 1, 'creationdate': '2026-10-21T17:39:43+01:00', 'source': 'resume.pdf',

[{'id': 'b6e97c8d-4a56-409a-8f64-45c179346450',
  'content': "Psiog Digital Feb 2022 – Sep 2022\nFull Stack Engineer Trainee Chennai, India\n• Completed a rigorous 6-month full stack development training programme.\n• Built a Movie Recommendation App using C#, ASP.NET, Angular, SQL and Entity Framework.\nPROJECTS\nAI-Generated Face Detection with Vision Transformers & Explainable AI | MSc research · Python, TensorFlow\n• Trained a Vision Transformer model to classify real versus AI-generated face images.\n• Applied explainability techniques to show which facial regions drove each prediction, making the model's decisions \ntransparent and auditable.\nLLM Chatbot with LangChain | Python, LangChain\n• Built a conversational assistant that uses LangChain to orchestrate LLM calls, prompt templates and conversation memory.\n• Designed prompts and response handling to keep answers relevant and accurate across multi-turn conversations.\nReview Sentiment Analysis | Python, NLP",
  'metadata': {

In [49]:
rag_retriever.retrieve("Byte-PPair Encoding (BPE)?", top_k=3, score_threshold=0.5)


🔹 Generating embeddings for 1 texts using model: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.61it/s]

✅ Generated embeddings with shape: (1, 384)
✅ Retrieved document 1: ID=c33f0d4e-0b01-42b0-bb17-7c9dbac4014f, Score=-0.2502, Distance=1.2502
Content Preview: lso power imge models ViT, speech Whisper), protein folding AlphFold) nd more.
2. The big picture
Every Trnsformer lnguge model follows the sme pipeline:
The pipeline of  decoder-only Tr...
Metadata: {'creationdate': '2026-10-22T10:53:25+00:00', 'title': 'Transformers', 'page': 2, 'page_label': '3', 'producer': 'Skia/PDF m141', 'moddate': '2026-10-22T10:53:25+00:00', 'subject': 'The architecture behind GPT, BERT, Claude and modern AI, built up from tokens and attention to a working GPT, Hugging Face models and the techniques used in production.', 'file_type': '.pdf', 'author': 'Study guide', 'creator': 'Chromium', 'source': '2-Transformers-Basic-to-Advanced.pdf', 'total_pages': 25}

✅ Retrieved document 2: ID=ecffdf74-b944-49f6-9fcf-341664de6b8e, Score=-0.4201, Distance=1.4201
Content Preview: Contents
Prt 1 Foundt

[{'id': 'c33f0d4e-0b01-42b0-bb17-7c9dbac4014f',
  'content': "\ue02clso power im\ue02cge models \ue081ViT\ue082, speech \ue081Whisper), protein folding \ue081Alph\ue02cFold) \ue02cnd more.\n2. The big picture\nEvery Tr\ue02cnsformer l\ue02cngu\ue02cge model follows the s\ue02cme pipeline:\nThe pipeline of \ue02c decoder-only Tr\ue02cnsformer such \ue02cs GPT. Everything in this guide expl\ue02cins one of these boxes.\n\ue06d. Tokeniz\ue02ction\nModels don't re\ue02cd ch\ue02cr\ue02ccters or whole words; they re\ue02cd tokens. Modern tokenizers use subword units:\ncommon words get \ue02c single token, while r\ue02cre words \ue02cre split into pieces. This keeps the voc\ue02cbul\ue02cry\nm\ue02cn\ue02cge\ue02cble \ue081\ue06d0,000 to 200,000 tokens) while still h\ue02cndling \ue02cny word, including typos \ue02cnd new n\ue02cmes.\nThe most common \ue02clgorithm is Byte-P\ue02cir Encoding \ue081BPE\ue082: st\ue02crt from single ch\ue02cr\ue02ccters, then repe\ue02ctedly\nmerge the most fr

### Retrieval Step

Integration VectorDB context pipeline with LLM Output

In [50]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()
# Loadingt the API keys from the .env file and setting them as environment variables
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model= init_chat_model("gpt-4.1")
model
llm = model


In [51]:
# Simple RAG Function : Retreive context + generate reponse
def rag_simple(query,retriever,llm,top_k=3,score_threshold=0.5):
    """
    A simple RAG function that retrieves context and generates a response using a language model.

    Args:
        query (str): The input query string.
        retriever (RAGRetriever): The RAG retriever instance to use for retrieving documents.
        llm: The language model instance to use for generating responses.
        top_k (int): Number of top documents to retrieve.
        score_threshold (float): Minimum similarity score for a document to be included in the results.

    Returns:
        str: The generated response from the language model based on the retrieved context.
    """
    # Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)

    # Combine the content of the retrieved documents into a single context string
    context = "\n\n".join([doc["content"] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        print("⚠️ No relevant documents found for the query. Proceeding with an empty context.")

    # Generate a response using the language model
    prompt = """
    use the following context to answer the query. If the context is not sufficient, answer based on your knowledge.
    context: {context}
    query: {query}
    Answer : 
    """
    response = llm.invoke(prompt.format(context=context, query=query))

    return response.content

In [52]:
answer = rag_simple("What is the RAG framework?", rag_retriever, llm, top_k=3, score_threshold=0.5)
print("Generated Answer:\n", answer)

🔹 Generating embeddings for 1 texts using model: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.18it/s]

✅ Generated embeddings with shape: (1, 384)
✅ Retrieved document 1: ID=5d4f2059-81d5-4f2f-ad28-4f3c2c063a74, Score=-0.1131, Distance=1.1131
Content Preview: Contents
Prt 1 Foundtions
1. Why RAG?
2. The RAG pipeline
. A complete RAG system from scrtch
Prt 2 Building blocks
. Loding documents
5. Chunking
. Embedding nd storing
7. Prompting for gr...
Metadata: {'author': 'Study guide', 'creationdate': '2026-10-22T10:53:26+00:00', 'page_label': '2', 'producer': 'Skia/PDF m141', 'title': 'Retrieval-Augmented Generation (RAG)', 'creator': 'Chromium', 'moddate': '2026-10-22T10:53:26+00:00', 'total_pages': 21, 'page': 1, 'source': '3-RAG-Basic-to-Advanced.pdf', 'subject': 'How to make an LLM answer from your own documents, from a 40-line prototype to hybrid search, reranking, evaluation, agents and production.', 'file_type': '.pdf'}

✅ Retrieved document 2: ID=3c1f8b69-7cb5-4be3-ae49-ad6d35cacec4, Score=-0.2213, Distance=1.2213
Content Preview: RAG or fine-tuning?
Need Best pproch

Generated Answer:
 **Answer:**  
The RAG (Retrieval-Augmented Generation) framework is a method for building systems where a language model answers questions using information retrieved from a collection of documents. It works in two phases:

1. **Indexing**: Documents are processed ("loaded"), broken into chunks, transformed into embeddings, and stored in a way that allows them to be quickly retrieved later. This phase occurs ahead of time and is repeated whenever documents change.
2. **Querying**: When a user asks a question, the system retrieves relevant chunks from the indexed documents, and the language model generates an answer using this retrieved information (grounded answers).

**Key Features of RAG:**
- Allows language models to answer questions with up-to-date and specific information from external documents, without needing to retrain.
- Supports immediate updates by simply changing the source documents.
- Enables citation and evidence for answers by providing references to

### Enhanced RAG pipeline Features

In [56]:
def rag_advanced(query, retriever, llm, top_k=3, score_threshold=0.2,return_context=False):
    """
    An advanced RAG function that retrieves context and generates a response using a language model.
    This function provides detailed information about the retrieved documents.

    Args:
        query (str): The input query string.
        retriever (RAGRetriever): The RAG retriever instance to use for retrieving documents.
        llm: The language model instance to use for generating responses.
        top_k (int): Number of top documents to retrieve.
        score_threshold (float): Minimum similarity score for a document to be included in the results.

    Returns:
        str: The generated response from the language model based on the retrieved context.
    """
    # Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    if not retrieved_docs:
        return {'answer': "⚠️ No relevant documents found for the query. Proceeding with an empty context.", 'context': "",'sources': [],'confidence': 0.0}

    # Combine the content of the retrieved documents into a single context string
    context = "\n\n".join([doc["content"] for doc in retrieved_docs]) if retrieved_docs else ""
    sources = [{
        'source': doc["metadata"].get("source_file", doc["metadata"].get("source", "Unknown")),
        'page': doc["metadata"].get("page_number", "Unknown"),
        'score': doc["similarity_score"],
        'preview': doc["content"][:200]   + '...'# First 200 characters of the document
    } for doc in retrieved_docs]
    confidence = np.mean([doc["similarity_score"] for doc in retrieved_docs]) if retrieved_docs else 0.0
    if not context:
        print("⚠️ No relevant documents found for the query. Proceeding with an empty context.")

    # Generate a response using the language model
    prompt = """
    Use the following context to answer the query. If the context is not sufficient, answer based on your knowledge.
    
    Context: {context}
    
    Query: {query}
    
    Answer:
    """
    response = llm.invoke(prompt.format(context=context, query=query))
    output = {
        'answer': response.content,
        'context': context,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
        return output

    return response.content

In [59]:
answer = rag_advanced("What is the transformers and attention?", rag_retriever, llm, top_k=3, score_threshold=0.5,return_context=True)
print("Answer:\n", answer['answer'])
print("Context:\n", answer['context'])
print("Sources:\n", answer['sources'])
print("Confidence:\n", answer['confidence'])

🔹 Generating embeddings for 1 texts using model: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.96it/s]

✅ Generated embeddings with shape: (1, 384)
✅ Retrieved document 1: ID=dc0f5942-a87e-4ac6-b5af-2515535fd259, Score=-0.2736, Distance=1.2736
Content Preview: print("Output:", out.shape, "(same shape as the input)")
Output
Input:  (6, 16)
Attention weights per head: (4, 6, 6) (heads, tokens, tokens)
Output: (6, 16) (same shape as the input)
Transformers 10 ...
Metadata: {'moddate': '2026-10-22T10:53:25+00:00', 'source': '2-Transformers-Basic-to-Advanced.pdf', 'creator': 'Chromium', 'subject': 'The architecture behind GPT, BERT, Claude and modern AI, built up from tokens and attention to a working GPT, Hugging Face models and the techniques used in production.', 'producer': 'Skia/PDF m141', 'total_pages': 25, 'file_type': '.pdf', 'page': 9, 'creationdate': '2026-10-22T10:53:25+00:00', 'author': 'Study guide', 'page_label': '10', 'title': 'Transformers'}

✅ Retrieved document 2: ID=e65ac9fe-ebb1-4a87-8c93-a504068dc981, Score=-0.3269, Distance=1.3269
Content Preview: Key ide: the totl cos

Answer:
 **Transformers** are a type of neural network architecture introduced in 2017, widely used for tasks involving sequential data, like natural language processing (NLP). Unlike earlier models such as RNNs or LSTMs, transformers rely almost entirely on a mechanism called **attention** to process input data in parallel, making them much faster and more effective for large-scale language tasks.

### Attention

**Attention** is a technique that allows the model to focus on different parts of the input sequence when generating an output. For each token (word or piece of input), attention assigns different weights to all other tokens—effectively letting the model "attend" to relevant words, regardless of their position in the sentence.

In transformers, attention is usually implemented using a mechanism called **self-attention**, where each word in a sentence gets a score for all other words, helping the model capture relationships and context.

#### Multi-head attention

Transformers

### RAG Advanced with Citation, History & Summarization

In [ ]:
## Should Implement it